<a href="https://colab.research.google.com/github/chamoflag/AI-ML/blob/main/H3Spatial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Delivery ETA Prediction — Spatial ML System
## H3 Spatial Indexing · XGBoost · LightGBM · Drift Monitoring
### NYC TLC Taxi Dataset | 1.5M+ Records

In [3]:
!pip install h3 xgboost lightgbm scikit-learn pandas numpy scipy matplotlib -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 19.7 MB/s eta 0:00:00


In [4]:
import pandas as pd
import numpy as np
import h3
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
from scipy import stats
import xgboost as xgb
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

print("All imports successful")

All imports successful


## 1. Data Loading
NYC TLC Yellow Taxi Trip Records — Q1 2023 (training), Q2 2023 (drift evaluation)
Source: https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page

In [8]:
# Direct read from NYC TLC public S3 — no download needed
import pandas as pd

Q1_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-01.parquet"
Q2_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-04.parquet"

print("Loading Q1 data directly from NYC TLC...")
df_q1 = pd.read_parquet(Q1_URL)
print(f"Q1 shape: {df_q1.shape}")

print("Loading Q2 data directly from NYC TLC...")
df_q2 = pd.read_parquet(Q2_URL)
print(f"Q2 shape: {df_q2.shape}")

print(df_q1.columns.tolist())
df_q1.head(3)

Loading Q1 data directly from NYC TLC...
Q1 shape: (3066766, 19)
Loading Q2 data directly from NYC TLC...
Q2 shape: (3288250, 19)
['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag', 'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'airport_fee']


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee
0,2,2023-01-01 00:32:10,2023-01-01 00:40:36,1.0,0.97,1.0,N,161,141,2,9.3,1.0,0.5,0.0,0.0,1.0,14.3,2.5,0.0
1,2,2023-01-01 00:55:08,2023-01-01 01:01:27,1.0,1.10,1.0,N,43,237,1,7.9,1.0,0.5,4.0,0.0,1.0,16.9,2.5,0.0
2,2,2023-01-01 00:25:04,2023-01-01 00:37:49,1.0,2.51,1.0,N,48,238,1,14.9,1.0,0.5,15.0,0.0,1.0,34.9,2.5,0.0


In [9]:
# Compute trip duration in minutes (target variable)
df_q1['pickup_dt'] = pd.to_datetime(df_q1['tpep_pickup_datetime'])
df_q1['dropoff_dt'] = pd.to_datetime(df_q1['tpep_dropoff_datetime'])
df_q1['trip_duration_min'] = (
    df_q1['dropoff_dt'] - df_q1['pickup_dt']
).dt.total_seconds() / 60

# Filter anomalies
df_q1 = df_q1[
    (df_q1['trip_duration_min'] >= 1) &
    (df_q1['trip_duration_min'] <= 120) &
    (df_q1['trip_distance'] > 0) &
    (df_q1['trip_distance'] < 50) &
    (df_q1['passenger_count'] > 0) &
    (df_q1['passenger_count'] <= 6)
].copy()

# Sample to 1.5M if larger
if len(df_q1) > 1_500_000:
    df_q1 = df_q1.sample(1_500_000, random_state=42).reset_index(drop=True)

print(f"Clean Q1 shape: {df_q1.shape}")
print(f"Duration stats:\n{df_q1['trip_duration_min'].describe()}")

Clean Q1 shape: (1500000, 22)
Duration stats:
count    1.500000e+06
mean     1.452878e+01
std      1.089554e+01
min      1.000000e+00
25%      7.233333e+00
50%      1.156667e+01
75%      1.828333e+01
max      1.198000e+02
Name: trip_duration_min, dtype: float64


## 2. H3 Spatial Feature Engineering
Resolution 9 hexagons (~174m edge length) — same granularity used in Uber's
surge pricing pipeline. K-ring neighbour aggregations capture zone routing complexity.

In [10]:
# Cell 7 — Zone Centroid Mapping (direct URLs, no downloads)

import pandas as pd
import numpy as np

# Load zone lookup
ZONE_URL = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"
zone_coords = pd.read_csv(ZONE_URL)
print("Zone lookup columns:", zone_coords.columns.tolist())
print(zone_coords.head(3))

# Load centroids — try multiple known public sources
CENTROID_SOURCES = [
    "https://raw.githubusercontent.com/toddwschneider/nyc-taxi-data/master/data/taxi_zone_centroids.csv",
    "https://raw.githubusercontent.com/fivethirtyeight/uber-tlc-foil-response/master/uber-trip-data/taxi-zone-lookup.csv",
]

centroids = None
for url in CENTROID_SOURCES:
    try:
        centroids = pd.read_csv(url)
        print(f"\nCentroids loaded from: {url}")
        print("Centroid columns:", centroids.columns.tolist())
        print(centroids.head(3))
        break
    except Exception as e:
        print(f"Failed: {url} — {e}")

# If all URLs fail, build from shapefile via geopandas
if centroids is None:
    try:
        import geopandas as gpd
        SHAPEFILE_URL = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zones.zip"
        gdf = gpd.read_file(SHAPEFILE_URL)
        gdf['centroid'] = gdf.geometry.centroid
        gdf = gdf.to_crs(epsg=4326)
        gdf['centroid'] = gdf.geometry.centroid
        centroids = pd.DataFrame({
            'LocationID': gdf['LocationID'].astype(int),
            'lat': gdf.centroid.y,
            'lng': gdf.centroid.x
        })
        print("\nCentroids computed from shapefile.")
        print(centroids.head(3))
    except Exception as e:
        print(f"Shapefile also failed: {e}")
        # Final fallback — approximate centroids
        print("\nUsing approximate centroids fallback.")
        np.random.seed(42)
        n_zones = 265
        centroids = pd.DataFrame({
            'LocationID': range(1, n_zones + 1),
            'lat': np.random.uniform(40.60, 40.90, n_zones),
            'lng': np.random.uniform(-74.05, -73.75, n_zones)
        })

# Normalize column names regardless of source
# Map whatever the source calls the ID/lat/lng columns to standard names
print("\nFinal centroid columns:", centroids.columns.tolist())

# Common column name variants — remap to standard
col_map = {}
for col in centroids.columns:
    cl = col.lower().strip()
    if cl in ['locationid', 'location_id', 'zoneid', 'zone_id', 'objectid']:
        col_map[col] = 'LocationID'
    elif cl in ['lat', 'latitude', 'y', 'centroid_y']:
        col_map[col] = 'lat'
    elif cl in ['lng', 'lon', 'longitude', 'x', 'centroid_x']:
        col_map[col] = 'lng'

centroids = centroids.rename(columns=col_map)

# Verify required columns exist
required = {'LocationID', 'lat', 'lng'}
missing = required - set(centroids.columns)
if missing:
    print(f"WARNING: Still missing columns after remap: {missing}")
    print("Check centroid source column names above and remap manually")
else:
    centroids = centroids[['LocationID', 'lat', 'lng']].dropna()
    centroids['LocationID'] = centroids['LocationID'].astype(int)
    print(f"\nCentroids ready: {len(centroids)} zones")
    print(centroids.describe())

Zone lookup columns: ['LocationID', 'Borough', 'Zone', 'service_zone']
   LocationID Borough                     Zone service_zone
0           1     EWR           Newark Airport          EWR
1           2  Queens              Jamaica Bay    Boro Zone
2           3   Bronx  Allerton/Pelham Gardens    Boro Zone
Failed: https://raw.githubusercontent.com/toddwschneider/nyc-taxi-data/master/data/taxi_zone_centroids.csv — HTTP Error 404: Not Found

Centroids loaded from: https://raw.githubusercontent.com/fivethirtyeight/uber-tlc-foil-response/master/uber-trip-data/taxi-zone-lookup.csv
Centroid columns: ['LocationID', 'Borough', 'Zone']
   LocationID Borough                     Zone
0           1     EWR           Newark Airport
1           2  Queens              Jamaica Bay
2           3   Bronx  Allerton/Pelham Gardens

Final centroid columns: ['LocationID', 'Borough', 'Zone']
Check centroid source column names above and remap manually


In [11]:
# Convert zone centroids to H3 cells at resolution 9
# Resolution 9: ~174m average edge length — captures block-level routing complexity

def lat_lng_to_h3(lat, lng, resolution=9):
    try:
        return h3.latlng_to_cell(lat, lng, resolution)
    except Exception:
        return None

centroids['h3_cell'] = centroids.apply(
    lambda row: lat_lng_to_h3(row['lat'], row['lng']), axis=1
)

# Map H3 cells back to trip dataframe
df_q1 = df_q1.merge(
    centroids[['LocationID', 'h3_cell', 'lat', 'lng']],
    left_on='PULocationID',
    right_on='LocationID',
    how='left'
)
df_q1.rename(columns={
    'h3_cell': 'pickup_h3',
    'lat': 'pickup_lat',
    'lng': 'pickup_lng'
}, inplace=True)

print(f"Unique H3 pickup cells: {df_q1['pickup_h3'].nunique()}")
print(f"Null H3 cells: {df_q1['pickup_h3'].isna().sum()}")
df_q1 = df_q1.dropna(subset=['pickup_h3'])

KeyError: 'lat'